# YouTube-Domain Fine-Tuning for All Emotion Benchmark Models

This notebook fine-tunes every seven-emotion benchmark model on the same YouTube-domain training split, then optionally pushes the adapted models to Hugging Face. It is designed for Google Colab GPU runtime.

Recommended runtime: **T4 GPU or better**. The RoBERTa-large model can be slow; run the smaller models first if time is limited.

## 1. Install dependencies

In [ ]:
!pip install -q "transformers>=4.41,<5.0.0" "datasets>=2.20" "accelerate>=0.30" "evaluate>=0.4" "scikit-learn>=1.5" "pandas>=2.2" "pyarrow>=15.0" "torch>=2.2" huggingface_hub

## 2. Clone or update the project repository

In [ ]:
import os
from pathlib import Path

repo_url = "https://github.com/chasezhang1999/youtube-emotion-analyzer.git"
repo_dir = Path("/content/youtube-emotion-analyzer")

if repo_dir.exists():
    %cd /content/youtube-emotion-analyzer
    !git pull
else:
    %cd /content
    !git clone {repo_url}
    %cd /content/youtube-emotion-analyzer

print("Working directory:", Path.cwd())

## 3. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Log in to Hugging Face

Run this before using `--push-to-hub`. The target namespace below is `chase1zhang`.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Dry-run the training configuration

This prints the model list, data paths, repo names, and hyperparameters without starting training.

In [ ]:
!python scripts/train_youtube_domain_all_models.py --model all --hub-namespace chase1zhang --dry-run

## 6. Train smaller / medium models first

These three models are the most important fair-comparison set before the large model. Each run uses the same YouTube-domain train and validation split.

In [ ]:
!python scripts/train_youtube_domain_all_models.py --model distilbert --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --fp16
!python scripts/train_youtube_domain_all_models.py --model samlowe_roberta --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --fp16
!python scripts/train_youtube_domain_all_models.py --model jhartmann_distilroberta --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 16 --max-length 128 --fp16

## 7. Optional: train RoBERTa-large

Run this only if Colab runtime and memory are sufficient. The command uses a smaller per-device batch, fp16, and gradient accumulation for T4 memory.

In [ ]:
!python scripts/train_youtube_domain_all_models.py --model jhartmann_roberta_large --hub-namespace chase1zhang --push-to-hub --epochs 3 --batch-size 4 --gradient-accumulation-steps 2 --max-length 128 --fp16

## 8. Inspect saved training metrics

In [ ]:
from pathlib import Path
import json

metrics_paths = sorted(Path("fine_tuned_model_files/youtube_domain_all_models").glob("*/youtube_domain_training_metrics.json"))
print("Metric files:", len(metrics_paths))
for path in metrics_paths:
    data = json.loads(path.read_text())
    metrics = data.get("metrics", {})
    print("\n", path.parent.name)
    print("repo:", data.get("repo_id"))
    print("accuracy:", round(metrics.get("eval_accuracy", 0), 4))
    print("macro_f1:", round(metrics.get("eval_macro_f1", 0), 4))
    print("elapsed_seconds:", data.get("elapsed_seconds"))

## 9. Next local step after Colab finishes

After the adapted models are pushed to Hugging Face, pull the repo locally and update `scripts/evaluate_updated_results.py` to include the new adapted model repos. Then regenerate CSV / Excel / report / PPT results.